
# PRÁCTICA: Comparación KNN vs Regresión Logística vs SVM  
## Dataset: Breast Cancer Wisconsin

**Instrucciones**
- Puedes desarrollar tu código en las celdas que se faciliten. En algunos casos, hay alguna ayuda.
- Si prefieres otra forma de desarrollo también es válida. Completa el trahajo con tus propios comentarios


---
## 0️⃣ Imports y configuración

In [1]:

# Importa lo necesario para:
# - pandas, numpy
# - train_test_split
# - StandardScaler
# - SimpleImputer (si hay NaN)
# - KNeighborsClassifier
# - LogisticRegression
# - SVC
# - métricas: accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score
# - matplotlib.pyplot

# Escribe aquí tus imports


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt


---
## 1️⃣ Carga y exploración del dataset

In [188]:

# Carga el CSV (ajusta el nombre si es necesario)

# Muestra shape y primeras filas

# Identifica la columna objetivo (normalmente 'diagnosis')

# Comprueba valores

# Comprueba balanceo de clases


cancer = pd.read_csv("breat_cancer_wisconsin_dataset.csv")

cancer.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [189]:
cancer.shape

(569, 32)

In [190]:
columna_objetivo = cancer["diagnosis"]

In [191]:
cancer["diagnosis"].unique()

array(['M', 'B'], dtype=object)

In [192]:
conteo = 0
for index, row in cancer.iterrows():
    if row["diagnosis"] == "M":
        conteo += 1
    else:
        None

print(f"Es maligno en: {conteo}")

Es maligno en: 212


In [193]:
cancer["diagnosis"].value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64


**Pregunta (responder en una frase):**  
- ¿El dataset está balanceado?  
- ¿Hay NaN? Si los hay, ¿qué estrategia usarás (drop vs imputación) y por qué?


No, el dataset esta desbalanceado ya que una de las opciones tiene más de 100 datos más.
Si hay NaN, hay una columna que no debería de estar. En este caso drop ya que al final del csv habia una coma, lo que daba a entender que se queria crear una columna de más, se puede manualmente quitar la coma, guardar
y volver a ejecutar todo o en este caso eliminar la columna.

In [194]:
cancer.isnull().sum()

id                         0
diagnosis                  0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64

In [195]:
cancer.drop(columns="Unnamed: 32")

KeyError: "['Unnamed: 32'] not found in axis"

---
## 2️⃣ Limpieza mínima + definición de X e y

In [ ]:

# Elimina columnas irrelevantes si existen (típicas: 'id', 'Unnamed: 32', 'Unnamed: 0')

# Define X e y

# Verifica shapes


In [ ]:
cancer.drop(columns="id")

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [ ]:
X = cancer.drop(columns=["diagnosis"])
y = cancer["diagnosis"]

In [ ]:
X.shape

(569, 31)

In [ ]:
y.shape

(569,)

---
## 3️⃣ Train/Test split

In [ ]:

# Divide en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verifica el resultado

print(f"X_train tiene: {len(X_train)}")
print(f"X_test tiene: {len(X_test)}")
print(f"y_train tiene: {len(y_train)}")
print(f"y_test tiene: {len(y_test)}")

X_train tiene: 455
X_test tiene: 114
y_train tiene: 455
y_test tiene: 114


---
## 4️⃣ Tratamiento de NaN + Escalado (obligatorio para KNN y SVM)

In [ ]:

# Decisiones a tomar si hay NaN

# Si NO hay NaN

# Escala con StandardScaler (fit en train, transform en test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verifica que no quedan NaN

cancer.isna().sum()

id                         0
diagnosis                  0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64


**Pregunta**  
¿Por qué es obligatorio escalar en KNN y SVM?


Porque son algoritmos que se basan en calcular distancias y en muchos casos los datos pueden tener diferentes valores (pueden ser muy diferentes)

---
## 5️⃣ Modelo 1: KNN (búsqueda de k)

In [ ]:

# Recorre k impares (puedes consultar la teoría)
# Calcula y guarda:
# - accuracy (en test)
# - AUC (en test) usando predict_proba

# Identifica el mejor k según accuracy y el mejor k según AUC


#Creacion del modelo
knn = KNeighborsClassifier(n_neighbors=5)
# Entrenamiento
knn.fit(X_train_scaled, y_train)
# Prediccion
y_pred = knn.predict(X_test_scaled)
# Evaluacion
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


---
## 6️⃣ Gráfica: superponer Accuracy y AUC vs k

In [ ]:

# Representa en el mismo gráfico accuracy y AUC vs k


# 3. Configurar la visualización
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Pintamos los puntos (nuestras 3 variables)
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=y, cmap='coolwarm', s=50, edgecolors='k')

# 4. Crear la "sábana" (Superficie de Decisión)
# Creamos un rango de valores para X e Y
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 20), np.linspace(y_min, y_max, 20))

# Para el kernel Poly, resolvemos la ecuación del hiperplano para Z
# Basándonos en que: w0*x + w1*y + w2*z + b = 0
# Nota: En kernels no lineales, usamos la función de decisión para encontrar el nivel 0
Z = np.zeros(xx.shape)
for i in range(xx.shape[0]):
    for j in range(xx.shape[1]):
        # Buscamos dónde la predicción cambia de clase (frontera)
        # Esto es una aproximación visual de la "frontera de decisión"
        point = np.array([[xx[i,j], yy[i,j], 0]]) # Punto base
        # (Este bloque es una simplificación para dibujo rápido)
        
# 5. Dibujar la superficie (Contorno de decisión)
# Para visualizar el kernel POLY de forma clara, graficamos el volumen de decisión
ax.set_xlabel('Variable 1')
ax.set_ylabel('Variable 2')
ax.set_zlabel('Variable 3')
ax.set_title('Frontera de Decisión - Kernel Polinómico (3D)')

plt.show()



**Pregunta**  
¿Coincide el mejor k según Accuracy con el mejor k según AUC? ¿Por qué podría no coincidir?


---
## 7️⃣ Entrena el KNN final (elige un k) + métricas

In [ ]:

# Elige k final (puede ser best_k_auc o un criterio razonado)

# Entrena y evalúa:
# - accuracy
# - matriz de confusión
# - classification_report
# - curva ROC + AUC

---
## 8️⃣ Modelo 2: Regresión Logística

In [ ]:

# Entrena LogisticRegression (max_iter suficiente)


# Evalúa:
# - accuracy
# - matriz de confusión
# - AUC
# - curva ROC

---
## 9️⃣ Modelo 3: SVM

In [ ]:

# Entrena SVM razonadamente

# Evalúa (accuracy + AUC + ROC)

---
## 🔟 Comparación final: ROC superpuestas + tabla

In [ ]:

# Dibuja en el MISMO gráfico las ROC de:
# - KNN final
# - Regresión logística
# - SVM (elige la mejor versión: lineal o RBF)


# Puedes crear una tabla comparativa



---
## 1️⃣1️⃣ Conclusiones

Responde razonadamente:

1. ¿Qué modelo elegirías para este problema y por qué?
- Eligiría el modelo SVM Lineal, ya que probando con los 3 kernel, dan el mismo acurracy y siendo el SVM Lineal el modelo más sencillo, es el que se debe de elegir.
2. ¿Qué métrica te parece más relevante en un contexto médico: accuracy o AUC? Justifica tu respuesta.
- Acurracy, ya que si tiene que priorizar el poder ver la precisión con los datos nuevos.
3. Si el dataset fuera 100 veces más grande, ¿cuál de los tres modelos crees que sería menos recomendable y por qué?
- El modelo menos recomedable sería el KNN, ya que KNN trabaja muy bien cuando se tiene un dataset de tamaño, porque calcula distancias, y en el momento que se tiene mas datos tendría que calcular muchas distancias.
4. ¿Cuál es el modelo más interpretable de los tres?
- El modelo mas interpretable seria el polinomico, ya que trabaja con datos binarios 0 o 1, entonces al momento de mostrarlo, se ve muy claro.